In [10]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'

# Kiểm tra và chọn thiết bị
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị đang sử dụng: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Thiết bị đang sử dụng: cpu


In [11]:
# Tải tokenizer PhoBERT (vinai/phobert-base)
MODEL = "vinai/phobert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
print(f"Tokenizer '{MODEL}' đã được tải thành công.")
print(f"Vocab size: {tokenizer.vocab_size}")

Tokenizer 'vinai/phobert-base' đã được tải thành công.
Vocab size: 64000


In [12]:
# Lớp gốc từ tuần 3 
class TextDataset(Dataset):
    """Lớp Dataset gốc từ tuần 3 - chỉ xử lý texts, không có labels."""

    def __init__(self, texts, tokenizer, max_length=256):
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length  # đồng bộ: max_length=256

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text   = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            padding='max_length',   # đồng bộ: padding='max_length'
            truncation=True,        # đồng bộ: truncation=True
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids':      inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
        }


# kế thừa TextDataset
class SentimentDataset(TextDataset):
    """
    Kế thừa TextDataset, bổ sung hỗ trợ nhãn (labels) cho bài toán
    phân loại cảm xúc 3 lớp: 0 = Tiêu cực, 1 = Trung tính, 2 = Tích cực.

    Tham số bổ sung
    ---------------
    labels : array-like, optional
        Mảng nhãn tương ứng với texts. Nếu None, dataset chạy ở chế độ
        inference (không trả về 'labels').
    """

    def __init__(self, texts, tokenizer, labels=None, max_length=256):
        # Gọi __init__ của lớp cha với cùng bộ tham số đồng bộ
        super().__init__(texts, tokenizer, max_length=max_length)
        self.labels = labels  # None → inference mode

    def __getitem__(self, idx):
        # Lấy input_ids + attention_mask từ lớp cha
        item = super().__getitem__(idx)

        # Đưa tensor lên đúng device
        item = {k: v.to(DEVICE) for k, v in item.items()}

        # Thêm nhãn nếu có
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long).to(DEVICE)

        return item


print("Định nghĩa lớp TextDataset (gốc) và SentimentDataset (kế thừa) thành công.")

Định nghĩa lớp TextDataset (gốc) và SentimentDataset (kế thừa) thành công.


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Chuyển về chữ viết thường (Lowercase)
    text = text.lower()
    
    # 2. Chuẩn hóa một số teencode, từ viết tắt học thuật phổ biến
    text = re.sub(r'\b(gv|thầy|cô)\b', 'giảng viên', text)
    text = re.sub(r'\b(sv)\b', 'sinh viên', text)
    text = re.sub(r'\b(slide|slđ)\b', 'bài giảng', text)
    text = re.sub(r'\b(đc)\b', 'được', text)
    text = re.sub(r'\b(khg|ko|kh|k)\b', 'không', text)
    text = re.sub(r'\b(ok|ổn)\b', 'tạm ổn', text)
    
    # 3. Làm sạch khoảng trắng thừa và ký tự đặc biệt nhiễu
    text = re.sub(r'[^\w\s,.\-+]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("✅ Đã khởi tạo hàm chuẩn hóa ngôn ngữ học clean_text!")

In [13]:
# Tải dữ liệu đã xử lý từ tuần 1 (EDA_Baseline)
train_df = pd.read_csv('../data/processed/train_cleaned.csv').dropna()
test_df  = pd.read_csv('../data/raw/test.csv').dropna()

print(f"Train: {len(train_df):,} mẫu | Phân bố nhãn:")
print(train_df['label'].value_counts().sort_index().to_string())
print(f"\nTest : {len(test_df):,} mẫu")

Train: 11,426 mẫu | Phân bố nhãn:
label
0    5325
1     458
2    5643

Test : 3,166 mẫu


In [14]:
# Tham số chung – đồng bộ theo yêu cầu
MAX_LENGTH = 256

# Dataset huấn luyện (có nhãn)
train_dataset = SentimentDataset(
    texts     = train_df['tokenized'].values,
    tokenizer = tokenizer,
    labels    = train_df['label'].values,
    max_length= MAX_LENGTH,
)

# Dataset kiểm tra (có nhãn – dùng cột comment vì test chưa được tokenize)
test_dataset = SentimentDataset(
    texts     = test_df['comment'].values,
    tokenizer = tokenizer,
    labels    = test_df['label'].values,
    max_length= MAX_LENGTH,
)

print(f"Train dataset: {len(train_dataset):,} mẫu")
print(f"Test  dataset: {len(test_dataset):,} mẫu")

# Kiểm tra một mẫu
sample = train_dataset[0]
print("\n── Mẫu đầu tiên ──")
for key, val in sample.items():
    print(f"  {key}: shape={val.shape}, dtype={val.dtype}, device={val.device}")

Train dataset: 11,426 mẫu
Test  dataset: 3,166 mẫu

── Mẫu đầu tiên ──
  input_ids: shape=torch.Size([256]), dtype=torch.int64, device=cpu
  attention_mask: shape=torch.Size([256]), dtype=torch.int64, device=cpu
  labels: shape=torch.Size([]), dtype=torch.int64, device=cpu


In [15]:
# Cấu hình DataLoader
BATCH_SIZE_TRAIN = 32   # batch_size=32 cho training
BATCH_SIZE_EVAL  = 16   # batch_size=16 cho evaluation/inference

# Số worker tối ưu
NUM_WORKERS = 0 if os.name == 'nt' else 2

train_loader = DataLoader(
    dataset    = train_dataset,
    batch_size = BATCH_SIZE_TRAIN,
    shuffle    = True,           # trộn dữ liệu mỗi epoch
    num_workers= NUM_WORKERS,
    pin_memory = torch.cuda.is_available(),  # tăng tốc CPU→GPU transfer
    drop_last  = False,
)

test_loader = DataLoader(
    dataset    = test_dataset,
    batch_size = BATCH_SIZE_EVAL,
    shuffle    = False,          # không trộn khi đánh giá
    num_workers= NUM_WORKERS,
    pin_memory = torch.cuda.is_available(),
    drop_last  = False,
)

print(f"Train DataLoader: {len(train_loader):,} batches và batch_size={BATCH_SIZE_TRAIN}")
print(f"Test  DataLoader: {len(test_loader):,} batches và batch_size={BATCH_SIZE_EVAL}")

Train DataLoader: 358 batches và batch_size=32
Test  DataLoader: 198 batches và batch_size=16


In [16]:
# Lấy một batch từ train_loader để xác nhận shapes & device
batch = next(iter(train_loader))

print("── Batch từ train_loader ──")
for key, val in batch.items():
    print(f"  {key:15s}: shape={str(val.shape):30s} dtype={val.dtype}  device={val.device}")

# Xác nhận padding đúng max_length
assert batch['input_ids'].shape[1] == MAX_LENGTH, (
    f"Lỗi padding: kỳ vọng seq_len={MAX_LENGTH}, nhận được {batch['input_ids'].shape[1]}"
)

print(f"\n Pipeline hoạt động đúng - seq_len={MAX_LENGTH}, device={DEVICE}")

── Batch từ train_loader ──
  input_ids      : shape=torch.Size([32, 256])          dtype=torch.int64  device=cpu
  attention_mask : shape=torch.Size([32, 256])          dtype=torch.int64  device=cpu
  labels         : shape=torch.Size([32])               dtype=torch.int64  device=cpu

 Pipeline hoạt động đúng - seq_len=256, device=cpu


In [17]:
# Lấy một batch từ test_loader
test_batch = next(iter(test_loader))

print("── Batch từ test_loader ──")
for key, val in test_batch.items():
    print(f"  {key:15s}: shape={str(val.shape):30s} dtype={val.dtype}  device={val.device}")

print(f"\n Test loader sẵn sàng - {len(test_loader):,} batches và batch_size={BATCH_SIZE_EVAL}")

── Batch từ test_loader ──
  input_ids      : shape=torch.Size([16, 256])          dtype=torch.int64  device=cpu
  attention_mask : shape=torch.Size([16, 256])          dtype=torch.int64  device=cpu
  labels         : shape=torch.Size([16])               dtype=torch.int64  device=cpu

 Test loader sẵn sàng - 198 batches và batch_size=16
